In [1]:
import os
from typing import TypedDict, Literal

from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END

In [2]:
class State(TypedDict):
    sender: str
    subject: str
    body: str
    category: str
    action: str

print("State defined!")

State defined!


In [3]:
email = {
    "sender": "hr@example.com",
    "subject": "Interview Invitation",
    "body": "We would like to schedule an interview with you.",
    "category": "",
    "action": ""
}

In [4]:
def classify_email(state: State):
    return {"category": "important"}

print("Email classified!")

Email classified!


In [5]:
result = classify_email(email)
print(result)

{'category': 'important'}


In [6]:
def draft_reply(state: State):
    return {"action": "Draft reply"}

def archive_email(state: State):
    return {"action": "Archive"}

def ignore_email(state: State):
    return {"action": "Ignore"}

print("Action nodes created!")

Action nodes created!


In [7]:
def route_email(state: State):
    if state["category"] == "important":
        return "important"
    elif state["category"] == "normal":
        return "normal"
    else:
        return "spam"

print("Routing function created!")

Routing function created!


In [8]:
graph = StateGraph(State)

# Add nodes
graph.add_node("classify", classify_email)
graph.add_node("draft_reply", draft_reply)
graph.add_node("archive", archive_email)
graph.add_node("ignore", ignore_email)

# Start with classification
graph.add_edge(START, "classify")

# Conditional routing
graph.add_conditional_edges(
    "classify",
    route_email,
    {
        "important": "draft_reply",
        "normal": "archive",
        "spam": "ignore"
    }
)

# End each branch
graph.add_edge("draft_reply", END)
graph.add_edge("archive", END)
graph.add_edge("ignore", END)

app = graph.compile()

print("Email Assistant graph compiled!")

Email Assistant graph compiled!


In [9]:
result = app.invoke(email)

print(result)

{'sender': 'hr@example.com', 'subject': 'Interview Invitation', 'body': 'We would like to schedule an interview with you.', 'category': 'important', 'action': 'Draft reply'}


In [10]:
normal_email = {
    "sender": "friend@example.com",
    "subject": "Weekend plans",
    "body": "Are you free this Saturday?",
    "category": "",
    "action": ""
}

normal_result = app.invoke(normal_email)

print(normal_result)

{'sender': 'friend@example.com', 'subject': 'Weekend plans', 'body': 'Are you free this Saturday?', 'category': 'important', 'action': 'Draft reply'}


In [11]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

print("Gemini connected!")

ValidationError: 1 validation error for ChatGoogleGenerativeAI
  Value error, API key required for Gemini Developer API. Provide api_key parameter or set GOOGLE_API_KEY/GEMINI_API_KEY environment variable. [type=value_error, input_value={'model': 'gemini-3.6-fla...': {}, 'base_url': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [12]:
from dotenv import load_dotenv
load_dotenv()

False

In [13]:
import os

print("Current folder:", os.getcwd())
print("API key loaded:", os.getenv("GEMINI_API_KEY") is not None)

Current folder: C:\Users\PC\Desktop\100 DAYS OF AI\WEEK 2\DAY 14
API key loaded: False


In [14]:
from dotenv import load_dotenv
import os

load_dotenv(r"C:\Users\PC\Desktop\100 DAYS OF AI\.env")

print("API key loaded:", os.getenv("GEMINI_API_KEY") is not None)

API key loaded: False


In [15]:
from dotenv import load_dotenv
import os

load_dotenv(r"C:\Users\PC\Desktop\100 DAYS OF AI\WEEK 1\DAY 1\.env")

print("API key loaded:", os.getenv("GEMINI_API_KEY") is not None)

API key loaded: True


In [16]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

print("Gemini connected!")

Gemini connected!


In [17]:
def classify_email(state: State):
    prompt = f"""
    Classify this email into exactly one category:
    important, normal, or spam.

    Sender: {state["sender"]}
    Subject: {state["subject"]}
    Body: {state["body"]}

    Return only the category name.
    """

    response = llm.invoke(prompt)
    category = response.content.strip().lower()

    return {"category": category}

In [18]:
result = classify_email(email)
print(result)

AttributeError: 'list' object has no attribute 'strip'

In [19]:
def classify_email(state: State):
    prompt = f"""
    Classify this email into exactly one category:
    important, normal, or spam.

    Sender: {state["sender"]}
    Subject: {state["subject"]}
    Body: {state["body"]}

    Return only the category name.
    """

    response = llm.invoke(prompt)

    category = response.content[0]["text"].strip().lower()

    return {"category": category}

In [20]:
result = classify_email(email)
print(result)

{'category': 'important'}


In [21]:
result = app.invoke(email)

print("Category:", result["category"])
print("Action:", result["action"])

Category: important
Action: Draft reply


In [22]:
spam_email = {
    "sender": "random@example.com",
    "subject": "You won a prize!",
    "body": "Click this link to claim your free money.",
    "category": "",
    "action": ""
}

result = app.invoke(spam_email)

print("Category:", result["category"])
print("Action:", result["action"])

Category: important
Action: Draft reply


In [23]:
def classify_email(state: State):
    prompt = f"""
    Classify the email into exactly ONE category:
    important, normal, or spam.

    Rules:
    - important: job offers, interviews, deadlines, bills, or other important requests
    - normal: ordinary personal or general emails
    - spam: scams, fake prizes, suspicious offers, free-money messages, or phishing attempts

    Sender: {state["sender"]}
    Subject: {state["subject"]}
    Body: {state["body"]}

    Return ONLY one word:
    important
    normal
    spam
    """

    response = llm.invoke(prompt)

    category = response.content[0]["text"].strip().lower()

    return {"category": category}

In [24]:
result = app.invoke(spam_email)

print("Category:", result["category"])
print("Action:", result["action"])

Category: important
Action: Draft reply


In [25]:
response = llm.invoke("""
Classify this email as exactly one of: important, normal, spam.

Email:
Subject: You won a prize!
Body: Click this link to claim your free money.

This is clearly a scam or suspicious prize message.

Return ONLY the word: spam
""")

print(response.content)

[{'type': 'text', 'text': 'spam', 'extras': {'signature': 'EqQHCqEHARFNMg9hq7YHeFlR4dbAlvyVt8iE1ri+ezMvOG4fxEZHZaPHw1bJxRQ5crm8VyHDK3pI3pySznlz7U4nrqXWwAMUyjqK0gObsYepru144/7MfB0PWeHxFAKjl8/XUmlkzCnN4rY8kFHrEEd/L1e2yAe4jRziUXkgjDb8m3A/qoU8i4MYpbI8Yj0shDXN6HFeikGmInVE6aUtbdyd7qWAsi4SMuY0nvsGbuIWoPrE8PGH0Y06yg6X1OWOsMLkrtEnT5KEpR9LUrJTRI7irf+hHBQmUEfQDUFDgVgbFdieHOyyGz19Y3Fw0+zokq1TLbOMUwbvmvaUHkDmIVaxt8rj7JwoHdPeXrGSCUWz24FKQ4wuvm+y/zBpHBguuV3ogFq730ZtbrIe9B1LDCO74KjAn5J0bfOAhz9VmzYAaD/PO6GYqgKMrIq9CTx1RFSHHG3B4nZROw6qkcBycDdvUcn35ivTXO6xhwecmniCgfIgdnytfWlTTDQ0HZLgKm9QFFg5foSzeow+UBM7SHpv7dw86bM81pPKIhvFmexTBUM7rWwh8jbn6CCQp633zYy3Ulnvxw2Ge+k+jO7nxdEYcCTtLPgYsuIHp7Jumq217KFoPnllPDrbAi32f5ZZphS+gEx3R+XQqLxIemw99hlKuTzHS+2cm4jOM+//NSoILgLKrrvh6aQQ3KVCxdGslvNrVyQhreJzKyZU3EJMOFiT5mxRdqjYN1nPOa3/oaNrasLA61nN/CtzsZZR75meRDgfzPAnmbkDZaJ8XO05v9osFOq/r2VTl5iu2ZHWhvPRPIODtGOvQzPZ5PJwynUmATaXuP1D8lOE9gYa7b2nE3h07AfBwdmUbxmCluaCF4Wb5nlDc7k4+KoZs8DvWDGvjXSTbxaM1SNXchAxlgKskRLPe4kvAV1iRBEGtp3Ot3n8C

In [26]:
app = graph.compile()

print("Email Assistant graph recompiled!")

Email Assistant graph recompiled!


In [27]:
result = app.invoke(spam_email)

print("Category:", result["category"])
print("Action:", result["action"])

Category: important
Action: Draft reply


In [28]:
graph = StateGraph(State)

graph.add_node("classify", classify_email)
graph.add_node("draft_reply", draft_reply)
graph.add_node("archive", archive_email)
graph.add_node("ignore", ignore_email)

graph.add_edge(START, "classify")

graph.add_conditional_edges(
    "classify",
    route_email,
    {
        "important": "draft_reply",
        "normal": "archive",
        "spam": "ignore"
    }
)

graph.add_edge("draft_reply", END)
graph.add_edge("archive", END)
graph.add_edge("ignore", END)

app = graph.compile()

print("Email Assistant graph rebuilt!")

Email Assistant graph rebuilt!


In [29]:
result = app.invoke(spam_email)

print("Category:", result["category"])
print("Action:", result["action"])

Category: spam
Action: Ignore


In [30]:
normal_email = {
    "sender": "friend@example.com",
    "subject": "Weekend plans",
    "body": "Are you free this Saturday?",
    "category": "",
    "action": ""
}

result = app.invoke(normal_email)

print("Category:", result["category"])
print("Action:", result["action"])

Category: normal
Action: Archive


## Conclusion

In this project, I rebuilt a simple Email Assistant workflow using LangGraph.

The workflow uses Gemini to classify an email as important, normal, or spam. A conditional routing function then sends the email to the appropriate branch.

### Workflow

Email
→ Gemini Classification
→ Conditional Routing
→ Draft Reply / Archive / Ignore
→ END

### Key Learning

- State stores the email information.
- Nodes perform individual tasks.
- Gemini can classify the email.
- Conditional edges allow the workflow to branch.
- Python handles the routing decision.
- Different email categories can trigger different actions.

In [31]:
%%writefile README.md
# Day 14: Project - Email Assistant in LangGraph

## What I Built

I rebuilt the basic logic of an Email Assistant using LangGraph and Gemini.

The assistant classifies emails into three categories:

- Important
- Normal
- Spam

Based on the classification, the workflow takes a different action.

## Workflow

```text
Email
  ↓
Gemini Classification
  ↓
Conditional Routing
  ├── Important → Draft Reply
  ├── Normal → Archive
  └── Spam → Ignore

Writing README.md
